<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign4/NLP_assign4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 - Sentiment Analysis
---

In [ ]:
import pandas as pd
import nltk

In [2]:
# Download and extract IMDb Large Movie Review Dataset

import os
import tarfile
import urllib.request

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
filename = "aclImdb_v1.tar.gz"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)  # ~80MB, may take a minute

if not os.path.exists("aclImdb"):
    with tarfile.open(filename) as tar:
        tar.extractall(filter='data')

In [ ]:
# Loading dataset into a Pandas DataFrame
def load_reviews(split):
    data = []
    for label in ['pos', 'neg']:
        folder = os.path.join('aclImdb', split, label)
        for fname in os.listdir(folder):
            if fname.endswith('.txt'):
                with open(os.path.join(folder, fname), encoding='utf-8') as f:
                    data.append((f.read(), label))
    return pd.DataFrame(data, columns=['review', 'sentiment'])

train_df = load_reviews('train')
test_df = load_reviews('test')

train_df.head()

,review,sentiment
0,This one tends to get slighted by a lot of cri...,pos
1,George Cukor directs this high quality story o...,pos
2,"That 70s Show is the best TV show ever, period...",pos
3,This film is a wonderful movie based on the li...,pos
4,These are one of the movies that don't require...,pos


In [4]:
print(f"Training shape: {train_df.shape}")
print(f"Testing shape: {train_df.shape}")

Training shape: (25000, 2)
Testing shape: (25000, 2)


In [5]:
train_df['sentiment'].value_counts()

,count
sentiment,
pos,12500
neg,12500


In [6]:
X_train, y_train = train_df['review'], train_df['sentiment']
X_test, y_test = test_df['review'], test_df['sentiment']

In [ ]:
# Removing HTML tags from the reviews
import re

def clean_html(text):
    return re.sub(r"<[^>]+>", " ", text)

X_train = X_train.apply(clean_html)
X_test = X_test.apply(clean_html)

assert not X_train.str.contains("<br", regex=False).any(), "HTML tags still present after cleaning"
assert not X_test.str.contains("<br", regex=False).any(), "HTML tags still present after cleaning"
print("HTML markup removed. Example cleaned review:\n", X_train.iloc[0][:300])

HTML markup removed. Example cleaned review:
 This one tends to get slighted by a lot of critics and Kurosawa fans, but I thought it was wonderful. It's an episodic multi-character study of Tokyo's poorest, who live in a city literally made from garbage. Though it looks like an A-Bomb just hit, the film has a sort of serene beauty thanks to the


Using current versions of scikit-learn, pandas, nltk/spaCy, and matplotlib/seaborn, solve the below questions.

## Q1: Text Vectorization
---


 - Convert text into numerical features using Bag-of-Words (CountVectorizer), TF-IDF (TfidfVectorizer), and modern vectorization tools.
    - Explain, in your own words, the difference between Bag-of-Words and TF-IDF representations. Which one do you expect to perform better for sentiment analysis, and why?

In [ ]:
# Init stopwords and lemmatizer for tokenization
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# keep negation words out of the stopword list, or bigrams like "not good" lose their "not"
negation_words = {"not", "no", "nor", "never", "none", "cannot", "without"}
stop_words_nltk = set(stopwords.words("english")) - negation_words
lemmatizer = WordNetLemmatizer()

def lemma_tokenizer(text):
    words = word_tokenize(text.lower())
    return [lemmatizer.lemmatize(w) for w in words if w.isalpha() and w not in stop_words_nltk]

# sanity check: negation words must survive the stopword filter
assert lemma_tokenizer("This was not good.") == ["not", "good"]

In [9]:
# Bag of Words with bigrams capturing negation
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(tokenizer=lemma_tokenizer, token_pattern=None, ngram_range=(1, 2), max_features=10000)
X_train_cv = count_vec.fit_transform(X_train)
X_test_cv = count_vec.transform(X_test)

train_dtm = pd.DataFrame(X_train_cv.toarray(), columns=count_vec.get_feature_names_out())
train_dtm.index = X_train.index
train_dtm.head()

,aaron,abandon,abandoned,abc,ability,able,able get,able see,abomination,abortion,...,zero,zizek,zodiac,zombi,zombie,zombie film,zombie movie,zone,zoom,zorro
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# TF-IDF with bigrams capturing negation
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(tokenizer=lemma_tokenizer, token_pattern=None, ngram_range=(1, 2), max_features=10000)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

train_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
train_tfidf.index = X_train.index
train_tfidf.head()

,aaron,abandon,abandoned,abc,ability,able,able get,able see,abomination,abortion,...,zero,zizek,zodiac,zombi,zombie,zombie film,zombie movie,zone,zoom,zorro
0,0.0,0.0,0.150368,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Bag-of-Words just counts how many times each word shows up in a review. Every word gets equal weight, so common words like "movie" or "film" end up with high counts even though they show up in almost every review and don't help separate positive from negative. TF-IDF fixes this by scaling a word's count down if it appears in a lot of documents. A word that shows up a lot in one review but rarely across the whole dataset gets a high score, while a word that's everywhere gets pushed toward zero. So TF-IDF is Bag-of-Words plus a penalty for being too common.

I'd expect TF-IDF to do slightly better for sentiment analysis. Sentiment mostly comes from a handful of strong opinion words in a review, like "great", "terrible", "boring", or "brilliant", and these aren't always the most frequent words in the document. TF-IDF gives more weight to distinctive words like these instead of generic review vocabulary that shows up everywhere, so a model trained on it should pick up the right signal faster, especially Logistic Regression. Naive Bayes already models word probabilities per class, so the gap between BoW and TF-IDF should be smaller there, but TF-IDF should still help by muting uninformative high frequency words.

## Q2: Word Cloud
---

 - Generate a word cloud for positive reviews and negative reviews. Comment on the most frequent words observed in each class.

In [12]:
# Init fresh lemmatized tokenization for word cloud, with negation words in stopword list
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words_cloud = set(stopwords.words("english"))
cloud_lemmatizer = WordNetLemmatizer()

def cloud_tokenizer(text):
    words = word_tokenize(text.lower())
    return [cloud_lemmatizer.lemmatize(w) for w in words if w.isalpha() and w not in stop_words_cloud]

In [13]:
# Counting frequencies of lemmatized words
from collections import Counter

pos_words = Counter()
for review in X_train[y_train == "pos"]:
    pos_words.update(cloud_tokenizer(review))

neg_words = Counter()
for review in X_train[y_train == "neg"]:
    neg_words.update(cloud_tokenizer(review))

print(f"Unique lemmatized words: {len(pos_words)} positive, {len(neg_words)} negative")

Unique lemmatized words: 47731 positive, 46345 negative


In [ ]:
# Score each word by how disproportionately it favors this class over the other, weighted by its own frequency so rare words can't get an inflated score from noise
import math

def distinctiveness_scores(freq_this, freq_other, min_count=5, smoothing=1):
    scores = {}
    for word, count in freq_this.items():
        if count < min_count:
            continue
        other_count = freq_other.get(word, 0)
        skew = math.log((count + smoothing) / (other_count + smoothing))
        score = count * skew
        if score > 0:
            scores[word] = score
    return scores

pos_scores = distinctiveness_scores(pos_words, neg_words)
neg_scores = distinctiveness_scores(neg_words, pos_words)

assert pos_scores and neg_scores, "expected at least one distinctive word per class"
assert all(s > 0 for s in pos_scores.values()) and all(s > 0 for s in neg_scores.values())

In [ ]:
# Making word cloud for positive and negative reviews, sized by class-distinctiveness score
from wordcloud import WordCloud
import matplotlib.pyplot as plt

pos_wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="Dark2",
    random_state=42
).generate_from_frequencies(pos_scores)
neg_wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="viridis",
    random_state=42
).generate_from_frequencies(neg_scores)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
for ax, wc, title in zip([ax1, ax2], [pos_wc, neg_wc], ["Positive Reviews", "Negative Reviews"]):
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

<!-- TODO -->

## Q3: Sentiment Classification
---

 - Train and evaluate a Naive Bayes classifier and a Logistic Regression classifier for sentiment classification.

## Q4: Evaluation Metrics
---

 - Compare the two models using standard evaluation metrics. Build a comparison table summarizing Accuracy, Precision, Recall, F1-score, and training time for:
    - Naive Bayes + Bag-of-Words
    - Naive Bayes + TF-IDF
    - Logistic Regression + TF-IDF

## Review
---

Based on your results, answer the following:
<ol type="a">
    <li>Which model performed better overall?</li>
    <li>Which model is more interpretable, and why?</li>
    <li>Which model would you deploy in a production system? Justify your answer.</li>
</ol>